In [39]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
df = pd.read_csv('random_state158.csv')
df.head()

,userId_x,movieId,rating,timestamp_x,userId_y,tag,timestamp_y,title,genres,imdbId,tmdbId
0,17682,31433,3.5,1544094795,8360,hot guys,1289426369,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0
1,17682,31433,3.5,1544094795,23053,family,1602746509,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0
2,17682,31433,3.5,1544094795,23053,romance,1602746506,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0
3,17682,31433,3.5,1544094795,60001,watched 2007,1187719211,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0
4,17682,31433,3.5,1544094795,70971,Debra Messing,1286208696,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103531029 entries, 0 to 103531028
Data columns (total 11 columns):
 #   Column       Dtype  
---  ------       -----  
 0   userId_x     int64  
 1   movieId      int64  
 2   rating       float64
 3   timestamp_x  int64  
 4   userId_y     int64  
 5   tag          object 
 6   timestamp_y  int64  
 7   title        object 
 8   genres       object 
 9   imdbId       int64  
 10  tmdbId       float64
dtypes: float64(2), int64(6), object(3)
memory usage: 8.5+ GB


In [22]:
# this line creates std dev  just by n, in this sense (n - 0 = n) as in n-ddof degrees of freedom=0
user_stats = (df.groupby('userId_x').agg(user_mean=('rating', 'mean'),user_std=('rating', lambda x: x.std(ddof=0))).reset_index())

# this line uses std dev with denominator (n-1), this creates more nans for single raters as in n-1, becomes 1-1 so denominator becomes 0
    #user_stats = df.groupby('userId_x')['rating'].agg(['mean', 'std']).reset_index()
    #user_stats.rename(columns={'mean': 'user_mean', 'std': 'user_std'}, inplace=True)

In [41]:
display(user_stats.head())
num_zero_std = (user_stats['user_std'] == 0).sum()
print(f'Users with std dev = 0 are:{num_zero_std} out of: {len(user_stats)} or ({num_zero_std/len(user_stats):.2%})')
total_users = len(user_stats)

,userId_x,user_mean,user_std
0,7,3.0,0.0
1,9,4.0,0.0
2,10,3.0,0.0
3,12,2.0,0.0
4,16,2.0,0.0


Users with std dev = 0 are:48711 out of: 75598 or (64.43%)


In [27]:
ratings_per_user = df.groupby('userId_x')['movieId'].nunique().reset_index(name='num_ratings')
display(ratings_per_user.head())
single_raters = (ratings_per_user['num_ratings'] == 1).sum()
#print(single_raters)
print(f'Users with only one rating: {single_raters} ({single_raters/len(ratings_per_user):.2%})')

,userId_x,num_ratings
0,7,1
1,9,1
2,10,2
3,12,1
4,16,1


Users with only one rating: 44259 (58.55%)


### Real Ratings Method

Formula: user mean + (movie mean - global mean)


came from : rating for movie i by user u =global mean ratings +user bias + movie bias

user bias = user mean - global mean

movie biad = movie mean - global mean


"A bias baseline model for a recommender system using real, unnormalized ratings predicts an individual rating by accounting for the global average rating, the specific user's bias (tendency to rate higher or lower than average), and the specific item's bias (tendency to receive higher or lower ratings than average)"

In [28]:
df_copy = df.copy(deep=True) 

In [29]:
global_mean = df_copy['rating'].mean()

user_mean = df_copy.groupby('userId_x')['rating'].mean().rename('user_mean')
movie_mean = df_copy.groupby('movieId')['rating'].mean().rename('movie_mean')

df_copy = df_copy.merge(user_mean, on='userId_x').merge(movie_mean, on='movieId')
#predicting the movie rating by  user_mean + (rating movie mean-global_mean) global mean = mean of all ratings
df_copy['pred_rating'] = df_copy['user_mean'] + (df_copy['movie_mean'] - global_mean)


,userId_x,movieId,rating,timestamp_x,userId_y,tag,timestamp_y,title,genres,imdbId,tmdbId,user_mean,movie_mean,pred_rating
0,17682,31433,3.5,1544094795,8360,hot guys,1289426369,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0,0.677197,4.250,1.065591
1,17682,31433,3.5,1544094795,23053,family,1602746509,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0,0.677197,4.250,1.065591
2,17682,31433,3.5,1544094795,23053,romance,1602746506,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0,0.677197,4.250,1.065591
3,17682,31433,3.5,1544094795,60001,watched 2007,1187719211,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0,0.677197,4.250,1.065591
4,17682,31433,3.5,1544094795,70971,Debra Messing,1286208696,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0,0.677197,4.250,1.065591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103531024,185562,249,3.0,828293339,98941,CLV,1339262165,Immortal Beloved (1994),Drama|Romance,110116,13701.0,3.000000,3.675,2.813395
103531025,185562,249,3.0,828293339,105590,To See,1221068681,Immortal Beloved (1994),Drama|Romance,110116,13701.0,3.000000,3.675,2.813395
103531026,185562,249,3.0,828293339,116579,classical music,1290450836,Immortal Beloved (1994),Drama|Romance,110116,13701.0,3.000000,3.675,2.813395
103531027,185562,249,3.0,828293339,116579,Gary Oldman,1290450849,Immortal Beloved (1994),Drama|Romance,110116,13701.0,3.000000,3.675,2.813395


5

In [34]:
df_copy = df_copy.drop_duplicates(subset=['userId_x', 'movieId'])


In [37]:
df_copy.head(20).reset_index(drop=True)

,userId_x,movieId,rating,timestamp_x,userId_y,tag,timestamp_y,title,genres,imdbId,tmdbId,user_mean,movie_mean,pred_rating
0,17682,31433,3.5,1544094795,8360,hot guys,1289426369,"Wedding Date, The (2005)",Comedy|Romance,372532,6961.0,0.677197,4.250000,1.065591
1,192558,1754,5.0,1105656846,6296,Fallen Angel,1295747877,Fallen (1998),Crime|Drama|Fantasy|Thriller,119099,9411.0,4.094012,3.532258,3.764665
2,26576,5013,3.0,1377399565,1551,ensemble cast,1618090996,Gosford Park (2001),Comedy|Drama|Mystery,280707,5279.0,3.104545,3.700000,2.942940
3,35895,2723,4.0,1115821789,11089,parody,1443790132,Mystery Men (1999),Action|Comedy|Fantasy,132347,9824.0,4.593525,3.093023,3.824943
4,63586,5995,0.5,1447705624,1149,Roman Polanski,1424117703,"Pianist, The (2002)",Drama|War,253474,423.0,4.096680,4.128099,4.363174
5,166810,1359,2.5,1329114398,5951,Arnold Schwarzenegger,1168376593,Jingle All the Way (1996),Children|Comedy,116705,9279.0,4.068733,2.725806,2.932935
6,190512,6539,5.0,1213917989,268,Bechdel Test:Fail,1582063630,Pirates of the Caribbean: The Curse of the Bla...,Action|Adventure|Comedy|Fantasy,325980,22.0,4.020323,3.757447,3.916164
7,125619,1375,4.0,1348863961,4597,James Horner,1571742525,Star Trek III: The Search for Spock (1984),Action|Adventure|Sci-Fi,88170,157.0,3.666071,3.418605,3.223071
8,96651,36525,3.5,1446660758,3033,Reese Witherspoon,1215791976,Just Like Heaven (2005),Comedy|Fantasy|Romance,425123,9007.0,3.500000,3.558824,3.197218
9,155435,8807,3.5,1287193731,3059,Buddy movie,1473446165,Harold and Kumar Go to White Castle (2004),Adventure|Comedy,366551,11282.0,3.498294,3.612903,3.249591


In [40]:
rmse = np.sqrt(mean_squared_error(df_copy['rating'], df_copy['pred_rating']))
mae = mean_absolute_error(df_copy['rating'], df_copy['pred_rating'])

print(f"Baseline RMSE: {rmse:.3f}")
print(f"Baseline MAE: {mae:.3f}")

Baseline RMSE: 0.772
Baseline MAE: 0.559


In [61]:
def recommend_unrated_top_n(user_id, n=5):
    
    # look for the user_id on df copy, grab the user_id mean column, grab the first value
    user_mean = df_copy.loc[df_copy['userId_x']== user_id, 'user_mean'].iloc[0]
    
    
    # all movies df, we need user specific ratings for every movie
    all_movies = df_copy[['movieId', 'title', 'movie_mean']].drop_duplicates()

    
    # User specific prediction for all movies
    all_movies[str(user_id)+'_pred_rating'] = user_mean + (all_movies['movie_mean'] - global_mean)
   
    # Remove movies this user already rated, naming options: (seen-unseen, watched-unwatched, rated-unrated, old-new)
    rated_movies = df_copy[df_copy['userId_x'] == user_id]['movieId']
    #display(rated_movies)# user 189759 has 3 ratings
    unrated_movies = all_movies[~all_movies['movieId'].isin(rated_movies)]
    
    # sort user specific predictions in descending order, grab the top n
    top_n = unrated_movies.sort_values(str(user_id)+'_pred_rating', ascending=False).head(n)
    return top_n[['movieId', 'title', str(user_id)+'_pred_rating']]

recommend_unrated_top_n(189759, 5)

4.685619955979457


,movieId,title,189759_pred_rating
88015281,659,Purple Noon (Plein soleil) (1960),5.824015
52525714,8582,Manufacturing Consent: Noam Chomsky and the Me...,5.824015
50165382,145960,Skinwalker Ranch (2013),5.824015
50800118,4431,Rembrandt (1936),5.824015
94797840,79775,"Story of My Life, The (Mensonges et trahisons ...",5.824015


### Finding the most popular movies with minimum rating count

In [68]:
movie_stats = (df_copy.groupby(['movieId', 'title']).agg(movie_mean=('rating', 'mean'),rating_count=('rating', 'count')).reset_index())
#display(movie_stats)
min_ratings = 50
n = 50
popular_movies = movie_stats[movie_stats['rating_count'] >= min_ratings]
    

top_movies = popular_movies.sort_values('movie_mean', ascending=False).head(n)
display(top_movies)


,movieId,title,movie_mean,rating_count
4327,5971,My Neighbor Totoro (Tonari no Totoro) (1988),4.404412,68
277,318,"Shawshank Redemption, The (1994)",4.366883,462
672,858,"Godfather, The (1972)",4.355556,315
704,908,North by Northwest (1959),4.350000,90
936,1221,"Godfather: Part II, The (1974)",4.329843,191
8579,92259,Intouchables (2011),4.311475,61
922,1207,To Kill a Mockingbird (1962),4.310976,82
1530,2019,Seven Samurai (Shichinin no samurai) (1954),4.302083,96
11677,202439,Parasite (2019),4.296296,54
47,50,"Usual Suspects, The (1995)",4.294118,340


### Using the most popular movies (set of n count) movies, with user specific prediction ratings.



In [71]:
def recommend_popular_unrated_top_n(user_id, n=5):
    
    # look for the user_id on df copy, grab the user_id mean column, grab the first value
    user_mean = df_copy.loc[df_copy['userId_x']== user_id, 'user_mean'].iloc[0]
    
    
    # all movies df, we need user specific ratings for every movie
    user_top_movies = top_movies.copy()

    
    # User specific prediction for all movies
    user_top_movies[str(user_id)+'_pred_rating'] = user_mean + (user_top_movies['movie_mean'] - global_mean)
   
    # Remove movies this user already rated, naming options: (seen-unseen, watched-unwatched, rated-unrated, old-new)
    rated_movies = df_copy[df_copy['userId_x'] == user_id]['movieId']
    #display(rated_movies)# user 189759 has 3 ratings
    unrated_movies = user_top_movies[~user_top_movies['movieId'].isin(rated_movies)]
    
    # sort user specific predictions in descending order, grab the top n
    top_n = unrated_movies.sort_values(str(user_id)+'_pred_rating', ascending=False).head(n)
    return top_n[['movieId', 'title', str(user_id)+'_pred_rating']]

recommend_popular_unrated_top_n(189759, 20)

,movieId,title,189759_pred_rating
4327,5971,My Neighbor Totoro (Tonari no Totoro) (1988),5.228426
277,318,"Shawshank Redemption, The (1994)",5.190898
672,858,"Godfather, The (1972)",5.179570
704,908,North by Northwest (1959),5.174015
936,1221,"Godfather: Part II, The (1974)",5.153858
8579,92259,Intouchables (2011),5.135490
922,1207,To Kill a Mockingbird (1962),5.134990
1530,2019,Seven Samurai (Shichinin no samurai) (1954),5.126098
11677,202439,Parasite (2019),5.120311
47,50,"Usual Suspects, The (1995)",5.118132
